# Share a model across inference runs

Learn to bind a prepared model as a resource, submit several batches to one graph, and check predictions against direct PyTorch. You should know basic tensor operations and model inference. The default data is a synthetic lifecycle fixture, not MNIST or a model-quality benchmark.

## Before you begin

Run the cells in order in a fresh kernel. Keep the complete repository and bundled tutorial data together. The setup below locates your checkout; in Colab it accepts a complete source ZIP. The notebook imports the adjacent Python implementation, so operators and resources have one canonical source.


In [ ]:
from pathlib import Path
import importlib.util
import io
import subprocess
import sys
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Upload the complete Jayrun project ZIP, not just this notebook.
    if len(uploaded) != 1:
        raise ValueError("upload exactly one project ZIP")
    destination = Path("/content/jayrun_tutorial_source")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        if sum(item.file_size for item in archive.infolist()) > 500_000_000:
            raise ValueError("archive exceeds this notebook's extraction limit")
        for item in archive.infolist():
            path = (destination / item.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("unsafe archive path or symbolic link")
        archive.extractall(destination)
    candidates = [destination, *destination.iterdir()]
    ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run Jupyter from the extracted project, or upload the full project ZIP in Colab.")
sys.path.insert(0, str(ROOT))
print("Tutorial source:", ROOT)


## Prepare the environment

The next cell checks the packages this lesson uses. Locally, install missing tutorial packages before continuing. In Colab, it installs only missing non-PyTorch packages. For PyTorch lessons, provide a suitable existing PyTorch build; the cell does not replace it.


In [ ]:
required = {'packaging': 'packaging>=25', 'numpy': 'numpy', 'torch': 'torch'}
missing = [requirement for name, requirement in required.items() if importlib.util.find_spec(name) is None]
# Colab provides PyTorch. Never replace an existing CPU/CUDA installation here.
if "torch" in missing:
    raise RuntimeError("Install the PyTorch build appropriate for your hardware before running this lesson.")
if missing:
    if not IN_COLAB:
        raise RuntimeError(f"Install these optional tutorial dependencies first: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from packaging.version import Version
import packaging
if Version(packaging.__version__) < Version("25"):
    raise RuntimeError("This source archive requires packaging >= 25; update it explicitly before proceeding.")
import torch
torch.set_num_threads(1)  # Keep the small CPU demonstration bounded.


## 1. Inspect the resource binding

ModelResource owns loading, device selection, and teardown. InferBatch declares a ResourceField and consumes image values from each run. The builder binds the field before confirmation. The field allows parallel use because this example's model is used read-only; the flag does not add synchronization to an arbitrary model.


In [ ]:
import asyncio
import inspect
import tempfile
import torch
from pathlib import Path
from jayrun import ArtifactContext, Engine
from jayrun.context import ContextState
from tutorials import mnist_inference as lesson
from tutorials.mnist_data import Classifier, load_data

print(inspect.getsource(lesson.ModelResource))
print(inspect.getsource(lesson.build_graph))


## 2. Run two batches with one shared model

For this wiring exercise we save a randomly initialized classifier: its predictions are not useful accuracy evidence. The Engine can reuse the same compatible resource across both submissions. Each run supplies a different batch and keeps its own result. Teardown occurs through Engine ownership; merely finishing a batch does not unload the cache.


In [ ]:
def infer_two_batches():
    data = load_data(training_samples=64, validation_samples=32)
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(7)
        model = Classifier().eval()
    batches = data.validation_images.split(16)
    with tempfile.TemporaryDirectory() as temporary:
        checkpoint = Path(temporary) / "model.pt"
        torch.save(model.state_dict(), checkpoint)
        graph, images = lesson.build_graph(checkpoint)
        with Engine() as engine:
            runs = [engine.submit(graph, ArtifactContext({images: batch})) for batch in batches]
            predictions = []
            for run in runs:
                run.wait(timeout=30)
                if run.state is not ContextState.FINISHED:
                    raise RuntimeError(str(run.report.data.failure))
                predictions.extend(run.artifact(images).value)
    with torch.inference_mode():
        expected = model(data.validation_images).argmax(1).tolist()
    assert predictions == expected
    return {"contexts": len(runs), "predictions": len(predictions), "matches_direct": True}

shared = await asyncio.to_thread(infer_two_batches)
print(shared)
assert shared == {"contexts": 2, "predictions": 32, "matches_direct": True}


## 3. Run the complete inference demonstration

The complete demonstration prepares a centroid classifier and checks eight batches against direct predictions. contexts counts submitted batches; predictions counts individual samples; matches_direct_baseline checks integration equivalence. accuracy describes only the selected dataset/model pair.


In [ ]:
REAL_MNIST = False  # True explicitly downloads the real dataset.
DEVICE = "cpu"       # Use "cuda" only with an available GPU and suitable PyTorch.
result = await asyncio.to_thread(lesson.run_demo, real_mnist=REAL_MNIST, device=DEVICE)
print({key: result[key] for key in ("dataset", "device", "contexts", "predictions", "matches_direct_baseline", "accuracy")})
assert result["contexts"] == 8 and result["predictions"] == 256
assert result["matches_direct_baseline"]


## Try it yourself

Change split(16) to split(8) in the wiring exercise. Expect four contexts and still 32 predictions, with the same direct result; update the expected contexts count in the summary assertion. Reuse requires a compatible resource type, configuration, and sharing policy; separate Engines have separate ownership.

The complete real-data path creates a trusted state-dict checkpoint from its centroid model. Declared GPU capacity is admission accounting, not allocation or sharding. Requested unavailable CUDA is an error. Do not interpret the fixture's accuracy as MNIST accuracy.
